# Budgerigar：同音完整复读的 EnCodec 数据阶段

先验证 codec 自身能够重建输入波形，再缓存离散 token。真实下载、编码、解码和试听均在 Colab 执行。

In [ ]:
#@title 1. 更新项目并安装 codec 依赖
REPO_DIR='/content/Budgerigar'
from pathlib import Path
import subprocess,sys,importlib
if not Path(REPO_DIR).is_dir(): subprocess.run(['git','clone','--depth=1','https://github.com/DoctorAwe/Budgerigar.git',REPO_DIR],check=True)
else: subprocess.run(['git','-C',REPO_DIR,'pull','--ff-only'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',f'{REPO_DIR}[train]','transformers>=4.45','soundfile'],check=True)
sys.path.insert(0,REPO_DIR)
for name in [k for k in list(sys.modules) if k=='budgerigar' or k.startswith('budgerigar.')]: del sys.modules[name]
importlib.invalidate_caches()

In [ ]:
#@title 2. Drive 与原始 manifest
from google.colab import drive
drive.mount('/content/drive')
WORK_ROOT=Path('/content/drive/MyDrive/Budgerigar')
SOURCE_MANIFEST=WORK_ROOT/'manifests'/'cmu_arctic.jsonl'
assert SOURCE_MANIFEST.is_file(),SOURCE_MANIFEST
MODEL_ID='facebook/encodec_24khz'
BANDWIDTH=6.0 #@param {type:'number'}
REVISION='main'


In [ ]:
#@title 3. 8 条 codec 重建 smoke
from budgerigar.codec_features import extract_encodec_manifest,decode_encodec_payload
SMOKE_MANIFEST=WORK_ROOT/'manifests'/'cmu_arctic.encodec.smoke8.jsonl'
report=extract_encodec_manifest(SOURCE_MANIFEST,WORK_ROOT/'codec',SMOKE_MANIFEST,MODEL_ID,BANDWIDTH,REVISION,limit=8)
print(report)
import json,torchaudio
rows=[json.loads(x) for x in SMOKE_MANIFEST.read_text().splitlines() if x.strip()]
decoded,sr=decode_encodec_payload(rows[0]['codec_path'],MODEL_ID,REVISION)
RECON_PATH=WORK_ROOT/'codec'/'smoke_reconstruction.wav'
torchaudio.save(str(RECON_PATH),decoded.unsqueeze(0),sr)
print('试听重建:',RECON_PATH,decoded.shape,sr)
from IPython.display import Audio,display
display(Audio(str(rows[0]['audio_path']))); display(Audio(str(RECON_PATH)))

In [ ]:
#@title 4. 确认试听后提取全量 token
RUN_FULL=False #@param {type:'boolean'}
if not RUN_FULL: print('先试听 smoke；确认无严重失真后再设为 True。')
else:
    from budgerigar.codec_features import codec_fingerprint
    fp=codec_fingerprint(MODEL_ID,BANDWIDTH,REVISION)
    FULL_MANIFEST=WORK_ROOT/'manifests'/f'cmu_arctic.encodec.{fp}.jsonl'
    full=extract_encodec_manifest(SOURCE_MANIFEST,WORK_ROOT/'codec',FULL_MANIFEST,MODEL_ID,BANDWIDTH,REVISION)
    print(full)